In [9]:
import os
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [10]:
import wave

def check_wav_specs(file_path):
    with wave.open(file_path, 'rb') as wav_file:
        sample_rate = wav_file.getframerate()
        num_channels = wav_file.getnchannels()
        num_frames = wav_file.getnframes()

        print(f"Sample Rate: {sample_rate} Hz")
        print(f"Kanäle: {num_channels}")
        print(f"Frames: {num_frames}")
        print(f"Dauer: {num_frames / sample_rate:.2f} Sekunden")

check_wav_specs("../dataset/yes/00f0204f_nohash_0.wav")

Sample Rate: 16000 Hz
Kanäle: 1
Frames: 16000
Dauer: 1.00 Sekunden


In [11]:
# Check GPU
import torch

def check_gpu_status():
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available: {cuda_available}")

    if cuda_available:
        num_gpus = torch.cuda.device_count()
        print(f"Number of GPUs detected: {num_gpus}\n")
        for i in range(num_gpus):
            print(f"--- GPU {i} ---")
            print(f"Name: {torch.cuda.get_device_name(i)}")
            print(f"Compute Capability: {torch.cuda.get_device_capability(i)}")
            total_memory = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            print(f"Total Memory: {total_memory:.2f} GB")
    else:
        print("PyTorch cannot detect a compatible GPU. It will default to CPU.")

check_gpu_status()

CUDA Available: True
Number of GPUs detected: 1

--- GPU 0 ---
Name: NVIDIA GeForce RTX 5070 Laptop GPU
Compute Capability: (12, 0)
Total Memory: 7.96 GB


In [12]:
# 1. Configuration
DATA_DIR = "../dataset/"
CLASSES = ["yes", "no", "up", "down"]
TARGET_SAMPLE_RATE = 16000
NUM_SAMPLES = 16000 # 1 second of audio
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# 2. Data Preparation
def get_files_and_labels():
    file_paths = []
    labels = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

    for cls_name in CLASSES:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.exists(cls_dir):
            continue
        for file in os.listdir(cls_dir):
            if file.endswith(".wav"):
                file_paths.append(os.path.join(cls_dir, file))
                labels.append(class_to_idx[cls_name])

    return file_paths, labels


file_paths, labels = get_files_and_labels()
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

In [14]:
import soundfile as sf

class KeywordDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        clean_path = os.path.normpath(self.paths[idx])
        # Use soundfile directly to bypass ALL torchaudio/torchcodec backend errors
        waveform_np, sr = sf.read(clean_path, dtype="float32")
        waveform = torch.from_numpy(waveform_np)

        # Format shapes correctly
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        else:
            waveform = waveform.t()

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Pad or cut to exactly 1 second (16000 samples)
        if waveform.shape[1] > NUM_SAMPLES:
            waveform = waveform[:, :NUM_SAMPLES]
        elif waveform.shape[1] < NUM_SAMPLES:
            waveform = F.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

        # We return the RAW waveform. We will turn it into a Spectrogram on the GPU!
        return waveform, torch.tensor(self.labels[idx], dtype=torch.long)

# num_workers=4 makes the CPU load files in the background
# Change num_workers back to 0 so Windows doesn't freeze the notebook!
train_loader = DataLoader(KeywordDataset(train_paths, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(KeywordDataset(test_paths, test_labels), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)



In [15]:
# 4. Model Architecture (Added a 3rd Convolutional Layer to reach 95%)
class KeywordCNN(nn.Module):
    def __init__(self, num_classes):
        super(KeywordCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # <-- NEW LAYER

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc1 = nn.Linear(64 * 4 * 4, 128) # Updated input size for new layer
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x))) # Apply the new layer
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        logits = self.fc2(x)
        return logits

    def predict(self, x):
        logits = self.forward(x)
        return F.softmax(logits, dim=1)

model = KeywordCNN(num_classes=len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:
import time

# Create the Mel Spectrogram function AND put it immediately on the GPU!
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SAMPLE_RATE, n_mels=64
).to(DEVICE)

# 5. Training Loop
print(f"Training on {DEVICE}...")
for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    # enumerate() gives us the batch number
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        # Turn audio into Spectrograms instantly on the GPU
        with torch.no_grad():
            inputs = mel_transform(inputs)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        # ----- Print progress every 50 batches -----
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    train_acc = 100. * correct / total

    # Validation
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            # Spectrogram on GPU for test data too
            inputs = mel_transform(inputs)

            outputs = model(inputs)
            _, predicted = outputs.max(1)
            test_total += targets.size(0)
            test_correct += predicted.eq(targets).sum().item()

    test_acc = 100. * test_correct / test_total
    dur = time.time() - start_time

    # End of epoch summary with the time it took
    print(f"=== Epoch {epoch+1}/{EPOCHS} [{dur:.2f}s] Summary | Train Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% ===\n")

print("Training complete.")


Training on cuda...
Epoch [1/40] | Batch [50/238] | Loss: 1.2434
Epoch [1/40] | Batch [100/238] | Loss: 1.1185
Epoch [1/40] | Batch [150/238] | Loss: 0.9140
Epoch [1/40] | Batch [200/238] | Loss: 0.8005
=== Epoch 1/40 [4.74s] Summary | Train Loss: 1.1027 | Train Acc: 58.80% | Test Acc: 76.82% ===

Epoch [2/40] | Batch [50/238] | Loss: 0.5584
Epoch [2/40] | Batch [100/238] | Loss: 0.4985
Epoch [2/40] | Batch [150/238] | Loss: 0.5639
Epoch [2/40] | Batch [200/238] | Loss: 0.4015
=== Epoch 2/40 [4.76s] Summary | Train Loss: 0.6607 | Train Acc: 77.29% | Test Acc: 81.35% ===

Epoch [3/40] | Batch [50/238] | Loss: 0.5691
Epoch [3/40] | Batch [100/238] | Loss: 0.7833
Epoch [3/40] | Batch [150/238] | Loss: 0.4102
Epoch [3/40] | Batch [200/238] | Loss: 0.4881
=== Epoch 3/40 [4.86s] Summary | Train Loss: 0.4796 | Train Acc: 83.66% | Test Acc: 88.09% ===

Epoch [4/40] | Batch [50/238] | Loss: 0.4927
Epoch [4/40] | Batch [100/238] | Loss: 0.3028
Epoch [4/40] | Batch [150/238] | Loss: 0.4121
Epoch 